# 🌑 DarkForge-X: SHADOW-CORE INITIATED

## 🎯 Target: Super AI Engineer Season 6 - Constituency OCR Competition
### ⚔️ Weapon: PP-StructureV3 (Table & Layout Execution)
### ⚙️ Hardware Framework: P100 GPU Optimization
---
**Directive:** Extract structured voting data from Thai election documents (Form สส.6/1) using advanced OCR, table structure recognition, and chaotic post-processing logic.
**Rules of Engagement:** Total Annihilation of Levenshtein Distance.

In [5]:
# # [PHASE 1] BREACHING THE ENVIRONMENT - Installing Required Arsenal
# !pip install -q paddlepaddle-gpu -i https://mirror.baidu.com/pypi/simple
!pip install -q "paddleocr>=2.6.0.3"
# !pip install -q langchain langchain-community
# !pip install -q python-Levenshtein
# !pip install -q fuzzywuzzy
# !pip install -q opencv-python
# !pip install -q pandas numpy tqdm textdistance
# !pip install --upgrade pip

In [6]:
# [PHASE 2] IMPORTING DESTRUCTIVE PAYLOADS
import os
import cv2
import glob
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from paddleocr import PPStructure
import logging
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

# Suppress PaddleOCR debug logs for stealth mode
logging.getLogger().setLevel(logging.ERROR)

# CONFIGURATION NODE
IMAGE_DIR = './data/images/'
TEMPLATE_CSV = './sample_submission.csv'
CHECKPOINT_CSV = './checkpoint_darkforge.csv'
FINAL_SUBMISSION_CSV = './submission_darkforge.csv'
USE_GPU = True

print("[+] SHADOW-CORE: Dependencies Loaded. GPU Target Engaged.")

ImportError: cannot import name 'PPStructure' from 'paddleocr' (/home/drasogun/.venvs/vol3/lib/python3.13/site-packages/paddleocr/__init__.py)

In [ ]:
# [PHASE 3] LOGIC MANIPULATION - Custom Heuristics & Number Parsers

THAI_TO_ARABIC = {
    '๐': '0', '๑': '1', '๒': '2', '๓': '3', '๔': '4',
    '๕': '5', '๖': '6', '๗': '7', '๘': '8', '๙': '9'
}

def clean_and_convert_number(text):
    """
    Experimental Module: Purge all non-digit artifacts and convert Thai numerals to Arabic.
    """
    if not isinstance(text, str):
        text = str(text)
    
    # Replace Thai numerals
    for th, ar in THAI_TO_ARABIC.items():
        text = text.replace(th, ar)
    
    # Extract only digits
    digits = re.sub(r'\D', '', text)
    if digits == '':
        return '0'
    return digits

def map_predicted_party_to_template(extracted_party, template_parties, threshold=60):
    """
    Fuzzy Matching Algorithm: Matches corrupted OCR party names to the pure template.
    Returns the best match or None if below threshold.
    """
    if not extracted_party or pd.isna(extracted_party):
        return None
    best_match, score = process.extractOne(str(extracted_party), template_parties, scorer=fuzz.token_sort_ratio)
    if score >= threshold:
        return best_match
    return None

print("[+] LOGIC MODULES: Online and Armed.")

In [ ]:
# [PHASE 4] FIRING UP THE ENGINE - PP-StructureV3 Initialization
print("[!] Initializing PP-StructureV3 into GPU VRAM...")

# Utilizing Layout Analysis, Table Recovery, and highly accurate OCR
table_engine = PPStructure(
    layout=True,
    table=True,
    ocr=True,
    show_log=False,
    use_gpu=USE_GPU,
    lang='th',  # Crucial for Thai language
    det_db_box_thresh=0.3,
    rec_drop_score=0.1
)

print("[+] PP-StructureV3 Engine: WARMED UP. Awaiting Targets.")

In [ ]:
# [PHASE 5] TARGET RECONNAISSANCE - Grouping Document Pages

def group_documents(image_dir):
    all_images = glob.glob(os.path.join(image_dir, '*.png'))
    doc_groups = {}
    
    for img_path in all_images:
        filename = os.path.basename(img_path).replace('.png', '')
        # Filename logic: {type}_{province}_{constituency}[_pageX]
        parts = filename.split('_page')
        base_name = parts[0]
        
        if base_name not in doc_groups:
            doc_groups[base_name] = []
        doc_groups[base_name].append(img_path)
        
    # Sort pages within each document
    for base_name in doc_groups:
        doc_groups[base_name].sort()
        
    return doc_groups

doc_groups = group_documents(IMAGE_DIR)
print(f"[+] Discovered {len(doc_groups)} Unique Documents spanning {len(glob.glob(os.path.join(IMAGE_DIR, '*.png')))} Pages.")

# Load the target schema
try:
    submission_df = pd.read_csv(TEMPLATE_CSV)
    if 'party' in submission_df.columns:
        expected_parties = submission_df['party'].dropna().unique().tolist()
    else:
        expected_parties = []
        for template_id in submission_df['id'].values:
            parts = str(template_id).split('_')
            party = parts[-1]
            if party not in expected_parties:
                expected_parties.append(party)
    print(f"[+] Submission Template Loaded. Discovered {len(expected_parties)} Unique Party Signatures.")
except Exception as e:
    print("[-] Warning: sample_submission.csv not found or corrupted. Proceeding with raw extraction.", e)
    submission_df = pd.DataFrame()
    expected_parties = []

In [ ]:
# [PHASE 6] THE MAIN ASSAULT - Executing OCR Batch with Persistence

extracted_results = []
processed_count = 0

# Checkpoint recovery (Stealth persistence)
if os.path.exists(CHECKPOINT_CSV):
    print("[!] Found existing checkpoint. Resuming operations...")
    checkpoint_df = pd.read_csv(CHECKPOINT_CSV)
    extracted_results = checkpoint_df.to_dict('records')
    processed_docs = set([res['document_id'] for res in extracted_results])
    remaining_docs = {k: v for k, v in doc_groups.items() if k not in processed_docs}
else:
    remaining_docs = doc_groups

print(f"[!] Initiating Extraction on {len(remaining_docs)} remaining documents...")

for doc_id, pages in tqdm(remaining_docs.items(), desc="Infiltrating Documents"):
    doc_extracted_data = {}
    
    for page_path in pages:
        img = cv2.imread(page_path)
        if img is None:
            continue
            
        # Execute PP-Structure Attack
        try:
            result = table_engine(img)
        except Exception as e:
            print(f"[-] Engine failure on {page_path}: {e}")
            continue
        
        # Interpret the OCR output structure
        for region in result:
            if region['type'] == 'Table':
                if 'res' in region and isinstance(region['res'], dict) and 'html' in region['res']:
                    html_content = region['res']['html']
                    try:
                        dfs = pd.read_html(html_content)
                        for df in dfs:
                            for row_idx, row in df.iterrows():
                                text_row = ' '.join(str(val) for val in row.values if not pd.isna(val))
                                num_candidates = re.findall(r'\d+', text_row.replace(',', ''))
                                if num_candidates:
                                    vote_val = clean_and_convert_number(num_candidates[-1])
                                    matched_party = map_predicted_party_to_template(text_row, expected_parties)
                                    if matched_party:
                                        if matched_party not in doc_extracted_data or int(vote_val) > int(doc_extracted_data[matched_party]):
                                            doc_extracted_data[matched_party] = vote_val
                    except Exception as e:
                        pass
            
            # Fallback for plain text detection
            if 'res' in region and isinstance(region['res'], list):
                for line_box in region['res']:
                    text = line_box.get('text', '')
                    if any(char.isdigit() for char in text):
                        num_candidates = re.findall(r'\d+', text.replace(',', ''))
                        if num_candidates:
                            vote_val = clean_and_convert_number(num_candidates[-1])
                            matched_party = map_predicted_party_to_template(text, expected_parties)
                            if matched_party:
                                if matched_party not in doc_extracted_data or int(vote_val) > int(doc_extracted_data[matched_party]):
                                    doc_extracted_data[matched_party] = vote_val

    # Compile Results
    for party, vote in doc_extracted_data.items():
        extracted_results.append({
            'document_id': doc_id,
            'party': party,
            'predicted_votes': vote
        })
        
    processed_count += 1
    
    # Persistence Activation
    if processed_count % 50 == 0:
        pd.DataFrame(extracted_results).to_csv(CHECKPOINT_CSV, index=False)
        print(f"[+] Checkpoint Secured: {processed_count} documents breached.")

# Final Save
df_results = pd.DataFrame(extracted_results)
df_results.to_csv(CHECKPOINT_CSV, index=False)
print("[+] OCR Assault Complete. All Data Mined.")

In [ ]:
# [PHASE 7] DATA FORGING - Forging the Ultimate Submission File

if not submission_df.empty and not df_results.empty:
    print("[!] Weaving harvested data into the Submission Template Format...")
    
    # Reset baseline to 0
    submission_df['votes'] = '0'
    
    # Map constructed predictions
    result_dict = {}
    for _, row in df_results.iterrows():
        result_dict[(row['document_id'], row['party'])] = str(row['predicted_votes'])
        
    update_count = 0
    
    for idx, row in submission_df.iterrows():
        row_id = row['id']
        parts = str(row_id).split('_')
        if len(parts) >= 3:
            if 'party' in submission_df.columns:
                party_name = row['party']
                idx_party = str(row_id).rfind(party_name)
                if idx_party != -1:
                    doc_base = str(row_id)[:idx_party].rstrip('_')
                else:
                    doc_base = "_".join(parts[:-1])
            else:
                party_name = parts[-1]
                doc_base = "_".join(parts[:-1])
                
            # Inject forged vote
            predicted_vote = result_dict.get((doc_base, party_name))
            if predicted_vote:
                submission_df.at[idx, 'votes'] = predicted_vote
                update_count += 1

    submission_df.to_csv(FINAL_SUBMISSION_CSV, index=False)
    print(f"[+] Success! {update_count} rows successfully overridden.")
    print(f"[+] Output unleashed at: {FINAL_SUBMISSION_CSV}. Ready for Kaggle Dominance.")
else:
    print("[-] Post-processing failed. Check if data was extracted correctly.")